In [27]:
from typesafe_sdk import Choice, Noul, TypeSafeClient
import pandas as pd
from pathlib import Path

In [20]:
POD_ID = "a5rkuis6k2klwv"
KEV_PORT = 8008
URL=f"https://{POD_ID}-{KEV_PORT}.proxy.runpod.net"
client = TypeSafeClient(
    api_key="local",
    base_url=URL,
    model="kev-latest"
)
URL

'https://a5rkuis6k2klwv-8008.proxy.runpod.net'

In [39]:
REVIEWS = Path("../reseñas")
en_reviews = REVIEWS / "resenas_ingles.csv"
en = pd.read_csv(en_reviews)
example = en.iloc[2]
example["Review Text"]

"This place is near Fremont street. Had to pay for parking out front, but it was only $1. We ordered the French Toast donut and two creme brulee donuts. The French Toast donut was AMAZING! The creme brulee donuts were not very good. The sugary top was very hard and the doughnut was honestly just not good at all. The 4 star rating is more for the French Toast....the creme brulee would have gotten 1 star. I'd eat here again. Their menu was awesome and they still had options around 10:15am on a Saturday morning. Go early though, certain donuts sell out quickly!"

In [40]:
# Same 1-5 / N/A scale and rules as few_shot_etiquetado_resenas.md.
# The question names are the labels.db columns, so answers compare directly with human labels.
SCALE = {
    "1": "Very negative",
    "2": "Negative",
    "3": "Mixed or acceptable",
    "4": "Positive",
    "5": "Very positive",
    "N/A": "The review does not give enough evidence to rate this aspect",
}

COMMON = (
    " Rate only this aspect, independently of the others. Do not infer it from the review's "
    "overall tone, its enthusiasm, a general recommendation, or a star rating."
)

INSTRUCTIONS = {
    "food": (
        "How does the review rate the food and drinks: quality, taste, freshness, preparation, "
        "variety and portions? Repeat visits or recommending specific products are indirect "
        "evidence and support a moderately positive rating. Do not rate items the reviewer "
        "has not tried yet."
    ),
    "wait_time": (
        "How does the review rate the speed of service, ordering, preparation or delivery? "
        "Answer N/A unless the review gives evidence of actual times or delays. Being rushed "
        "by staff or a confusing order is not evidence of a short or long wait."
    ),
    "service": (
        "How does the review rate the staff: friendliness, professionalism, communication, "
        "willingness to help and how they treat customers? Answer N/A if staff are mentioned "
        "without their treatment of the customer being judged."
    ),
    "price": (
        "How does the review rate the prices and perceived value for money? A complaint about "
        "portions counts only if the review links the portions to what was paid."
    ),
    "experience": (
        "How does the review rate the atmosphere, comfort, location, opening hours, facilities "
        "and the overall experience of the visit?"
    ),
}

questions = {}

for dim, text in INSTRUCTIONS.items():
    questions[dim] = Choice(instructions=text + COMMON, criteria=SCALE)


response = client.system_one(state=example["Review Text"], questions=questions)

In [41]:
print(example["Review Text"])

This place is near Fremont street. Had to pay for parking out front, but it was only $1. We ordered the French Toast donut and two creme brulee donuts. The French Toast donut was AMAZING! The creme brulee donuts were not very good. The sugary top was very hard and the doughnut was honestly just not good at all. The 4 star rating is more for the French Toast....the creme brulee would have gotten 1 star. I'd eat here again. Their menu was awesome and they still had options around 10:15am on a Saturday morning. Go early though, certain donuts sell out quickly!


In [42]:
response.answers

{'food': ChoiceAnswer(type='choice', choice='4', confidence=0.4459, probabilities={'1': 0.0107, '2': 0.0273, '3': 0.2034, '4': 0.5382, '5': 0.0853, 'N/A': 0.135}),
 'wait_time': ChoiceAnswer(type='choice', choice='N/A', confidence=0.4093, probabilities={'1': 0.0271, '2': 0.0374, '3': 0.1725, '4': 0.1918, '5': 0.0634, 'N/A': 0.5078}),
 'service': ChoiceAnswer(type='choice', choice='N/A', confidence=0.7334, probabilities={'1': 0.0135, '2': 0.0165, '3': 0.0763, '4': 0.0822, '5': 0.0337, 'N/A': 0.7779}),
 'price': ChoiceAnswer(type='choice', choice='4', confidence=0.3234, probabilities={'1': 0.0122, '2': 0.0274, '3': 0.1909, '4': 0.4362, '5': 0.1419, 'N/A': 0.1915}),
 'experience': ChoiceAnswer(type='choice', choice='4', confidence=0.3679, probabilities={'1': 0.0072, '2': 0.0214, '3': 0.1627, '4': 0.4732, '5': 0.1286, 'N/A': 0.2068})}